# 03 – Multi-Objective Optimisation

This notebook uses the trained PINN checkpoint from `02_training.ipynb` to search for Pareto-optimal LPBF process parameters.

Two optimisers are available:

1. **NSGA-III** (`src/optimiser/nsga3.py`) – evolutionary many-objective optimisation.
2. **Bayesian optimisation** (`src/optimiser/bayesopt.py`) – sequential single-objective optimisation via Ax/BoTorch.

Both optimisers query the PINN surrogate at the origin `(x, y, z) = (0, 0, 0)` and final time `t = 1`, which approximates a representative coupon location.

In [ ]:
import os
import sys
from pathlib import Path

# Make the repository's src/ package importable from this notebook
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.insert(0, project_root)

# Scripts in src/ expect to be run from the repository root, so switch CWD
os.chdir(project_root)
print(f"Working directory: {os.getcwd()}")
print(f"Project root: {project_root}")

## 3.1 Locate the trained checkpoint and saved config

`PINNTrainer` creates a `data/models/latest` symlink pointing to the most recent training run. The run directory also contains `config.yaml`, which preserves any quick-demo overrides (e.g. smaller hidden layers) used during training. We use this saved config so the surrogate model architecture matches the checkpoint exactly.

In [ ]:
latest_link = Path('data/models/latest')
if not latest_link.exists():
    raise FileNotFoundError(
        "No 'data/models/latest' symlink found. Run 02_training.ipynb first."
    )

model_path = latest_link / 'checkpoints' / 'best_model.pt'
run_config_path = latest_link / 'config.yaml'
print(f"Latest checkpoint: {model_path.resolve()}")
print(f"Checkpoint exists: {model_path.exists()}")
print(f"Run config: {run_config_path.resolve()}")
print(f"Run config exists: {run_config_path.exists()}")

## 3.2 NSGA-III optimisation

The objectives are:

- Minimise residual stress
- Minimise porosity
- Maximise geometric accuracy (internally negated so NSGA-III minimises)

We use a tiny population and only a few generations for a quick demo.

In [ ]:
import yaml

from src.optimiser.nsga3 import NSGAOptimizer

# Load the config that was saved alongside the checkpoint; it already contains
# the same model architecture used during training.
with open(run_config_path, 'r') as f:
    opt_config = yaml.safe_load(f)

# Quick-demo overrides for the optimiser
opt_config['optimizer']['pop_size'] = 20
opt_config['optimizer']['n_gen'] = 5
opt_config['optimizer']['n_partitions'] = 4

# Write a temporary config file because NSGAOptimizer expects a path
tmp_config_path = os.path.join('notebooks', 'tmp_nsga_config.yaml')
with open(tmp_config_path, 'w') as f:
    yaml.dump(opt_config, f)

nsga_optimizer = NSGAOptimizer(tmp_config_path, str(model_path))
nsga_result = nsga_optimizer.optimize()

print(f"\nPareto front size: {len(nsga_result.X)}")
print(f"Results saved to: {nsga_optimizer.output_dir}")

## 3.3 Inspect NSGA-III results

The optimiser writes `pareto_solutions.csv` (human-readable) and `pareto_solutions.h5` (machine-readable). The objective values for `geometric_accuracy` are negated because NSGA-III minimises everything; we restore the sign below.

In [ ]:
import pandas as pd

pareto_csv = Path(opt_config['optimizer']['output_dir']) / 'pareto_solutions.csv'
df = pd.read_csv(pareto_csv)

# Restore the maximisation direction for geometric accuracy
if 'geometric_accuracy' in df.columns:
    df['geometric_accuracy'] = -df['geometric_accuracy']

print(f"First {min(5, len(df))} Pareto solutions:")
display(df.head())

df[['residual_stress', 'porosity', 'geometric_accuracy']].describe()

## 3.4 Optional: Bayesian single-objective optimisation

Bayesian optimisation is typically used for a single objective at a time. Here we minimise residual stress with a small number of trials for a quick illustration.

In [ ]:
try:
    from src.optimiser.bayesopt import BayesianOptimizer

    bayes_optimizer = BayesianOptimizer(tmp_config_path, str(model_path))
    best_params, best_values, _ = bayes_optimizer.run_single_objective_optimization(
        objective_idx=0,  # residual_stress
        n_trials=8,
    )
    print("\nBest parameters for minimising residual stress:")
    for param, value in best_params.items():
        print(f"  {param}: {value:.4f}")
except Exception as e:
    print(f"Bayesian optimisation skipped due to error: {e}")

## Next step

Proceed to `04_visualisation.ipynb` to generate animations and static plots from the results.